# Data Wrangling using Pandas

## What is Data Wrangling?

Data wrangling (also called data munging) is the process of transforming raw, messy data into a clean, structured format ready for analysis. In practice, data scientists spend **60-80% of their time** on this — it is the most important and most underestimated skill.

This notebook walks you through the entire wrangling pipeline: detecting and fixing missing values, removing duplicates, converting data types, standardizing text, reshaping data, combining multiple tables, and aggregating with GroupBy. Every technique is demonstrated on a realistic hospital patient dataset with intentionally messy data.

### The Healthcare Data Integration Disaster

A hospital merges patient data from three departments:

- **Emergency Department** entered data using **System A** (some fields optional)
- **Outpatient Clinic** used **System B** (different field names, different required fields)
- **Lab Results** came from **System C** (no patient name — only ID numbers)

**Result:** After merging, **35% of records had at least one missing field**.

**The danger:** A missing blood type in a patient record is not just a blank cell — it is a potential **transfusion error**. A duplicated patient ID means double billing — or worse, mixing up medication records.

| System | Department | Missing Fields | Issue |
|--------|-----------|---------------|-------|
| System A | Emergency | blood_group often blank | Optional field → missing data |
| System B | Outpatient | Different column names | Schema mismatch |
| System C | Lab Results | No patient name | ID-only → needs merge |

In [1]:
import pandas as pd

In [2]:
# ========================================
# SESSION DATASET: Hospital Patient Records
# This dataset has INTENTIONAL problems — your job is to find and fix them!
# Used throughout Sections 1-8
# ========================================

patients = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005',
                   'P006', 'P007', 'P008', 'P002', 'P010'],
    'name': ['Rajesh Kumar', ' Priya Sharma ', 'ANITA DESAI',
             'vikram patel', 'Sneha Iyer', None,
             'Meera Joshi', 'karan singh', 'Priya Sharma', 'Divya Rao'],
    'age': [45, 32, None, 28, 56, 41, None, 35, 32, 29],
    'blood_group': ['A+', 'B+', 'O-', 'AB+', 'A-',
                    'B+', 'O+', None, 'B+', 'A+'],
    'admission_date': ['2024-01-15', '2024-01-16', '2024-01-16',
                       '15-01-2024', '2024-01-17', '2024-01-18',
                       '2024-01-19', '2024-01-20', '2024-01-20', '2024-01-21'],
    'bill_amount': [15000, 22500, None, 18500, 20000.50, 'N/A',
                    17999.99, 19500, 22500, 16000],
    'department': ['Cardiology', 'cardiology', 'Orthopedics',
                   'orthopedics', 'Cardiology', 'CARDIOLOGY',
                   'Orthopedics', 'cardiology', 'cardiology', 'Cardiology']
})

print(f"Dataset loaded: {patients.shape[0]} rows x {patients.shape[1]} columns")
print(f"Columns: {patients.columns.tolist()}")

Dataset loaded: 10 rows x 7 columns
Columns: ['patient_id', 'name', 'age', 'blood_group', 'admission_date', 'bill_amount', 'department']


In [3]:
patients

,patient_id,name,age,blood_group,admission_date,bill_amount,department
0,P001,Rajesh Kumar,45.0,A+,2024-01-15,15000,Cardiology
1,P002,Priya Sharma,32.0,B+,2024-01-16,22500,cardiology
2,P003,ANITA DESAI,NaN,O-,2024-01-16,None,Orthopedics
3,P004,vikram patel,28.0,AB+,15-01-2024,18500,orthopedics
4,P005,Sneha Iyer,56.0,A-,2024-01-17,20000.5,Cardiology
5,P006,NaN,41.0,B+,2024-01-18,N/A,CARDIOLOGY
6,P007,Meera Joshi,NaN,O+,2024-01-19,17999.99,Orthopedics
7,P008,karan singh,35.0,NaN,2024-01-20,19500,cardiology
8,P002,Priya Sharma,32.0,B+,2024-01-20,22500,cardiology
9,P010,Divya Rao,29.0,A+,2024-01-21,16000,Cardiology


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Can You Spot All 7 Types of Problems?</strong><br><br>
    1. <strong>Missing values:</strong> <code>None</code> in name (row 5), age (rows 2, 6), blood_group (row 7), bill_amount (row 2)<br>
    2. <strong>String masquerading as missing:</strong> <code>'N/A'</code> in bill_amount (row 5) — looks missing but is actually a STRING<br>
    3. <strong>Duplicate record:</strong> Patient P002 (Priya Sharma) appears at rows 1 AND 8<br>
    4. <strong>Inconsistent name casing:</strong> 'Rajesh Kumar' vs 'ANITA DESAI' vs 'vikram patel' vs 'karan singh'<br>
    5. <strong>Leading/trailing whitespace:</strong> ' Priya Sharma ' has spaces on both sides<br>
    6. <strong>Inconsistent date formats:</strong> '2024-01-15' (ISO) vs '15-01-2024' (DD-MM-YYYY)<br>
    7. <strong>Inconsistent department casing:</strong> 'Cardiology' vs 'cardiology' vs 'CARDIOLOGY'
</div>

In [4]:
## Exploring the dataset.
print("==== Data Types ====")
print(patients.dtypes)

print("==== Dataset Info ====")
print(patients.info())

==== Data Types ====
patient_id            str
name                  str
age               float64
blood_group           str
admission_date        str
bill_amount        object
department            str
dtype: object
==== Dataset Info ====
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      10 non-null     str    
 1   name            9 non-null      str    
 2   age             8 non-null      float64
 3   blood_group     9 non-null      str    
 4   admission_date  10 non-null     str    
 5   bill_amount     9 non-null      object 
 6   department      10 non-null     str    
dtypes: float64(1), object(1), str(5)
memory usage: 692.0+ bytes
None



## Part 1: Handling Missing Values

### NaN vs None vs 'N/A' — Three Different Things

Before we start handling missing values, we need to understand that there are **three different representations** of "missing" that you will encounter:

- **`None`** — Python's built-in null object. Pandas converts this to `NaN` in numeric columns.
- **`np.nan`** — NumPy's "Not a Number" sentinel. This is the standard missing-value marker in Pandas.
- **`'N/A'`** (or `'null'`, `'None'`, `'-'`, `''`) — These are just regular **strings**. Pandas does NOT recognize them as missing unless you explicitly tell it to.

The first two are treated as missing by Pandas. The third is **not** — and that is where hidden bugs live.

In [5]:
# Three DIFFERENT representations of "missing" — only 2 are treated as NaN by Pandas
import numpy as np

val_none = None       # Python's None
val_nan = np.nan      # NumPy's Not a Number
val_na_str = 'N/A'    # Just a regular string!

print(f"None:  type = {type(val_none).__name__},  pd.isna() = {pd.isna(val_none)}")
print(f"NaN:   type = {type(val_nan).__name__}, pd.isna() = {pd.isna(val_nan)}")
print(f"'N/A': type = {type(val_na_str).__name__},   pd.isna() = {pd.isna(val_na_str)}")
print()
print("Key insight: Pandas treats None and NaN as missing, but 'N/A' is just text!")

None:  type = NoneType,  pd.isna() = True
NaN:   type = float, pd.isna() = True
'N/A': type = str,   pd.isna() = False

Key insight: Pandas treats None and NaN as missing, but 'N/A' is just text!


#### .isna() or .isnull() can be used to identify missing/null values

In [6]:
print(patients.isna())
print()
print(patients.isna().sum())
print()
print(patients.isnull().sum())
print()
print(f"Total Missing Values: {patients.isnull().sum().sum()}")

   patient_id   name    age  blood_group  admission_date  bill_amount  \
0       False  False  False        False           False        False   
1       False  False  False        False           False        False   
2       False  False   True        False           False         True   
3       False  False  False        False           False        False   
4       False  False  False        False           False        False   
5       False   True  False        False           False        False   
6       False  False   True        False           False        False   
7       False  False  False         True           False        False   
8       False  False  False        False           False        False   
9       False  False  False        False           False        False   

   department  
0       False  
1       False  
2       False  
3       False  
4       False  
5       False  
6       False  
7       False  
8       False  
9       False  

patient_id        0

In [7]:
percentage_missing = patients.isna().mean() * 100
for col, pct_missing in percentage_missing.items():
    status = 'OK' if pct_missing == 0 else f'{round(pct_missing, 2)}% Missing'
    print(f"{col:20s}: {status}")

patient_id          : OK
name                : 10.0% Missing
age                 : 20.0% Missing
blood_group         : 10.0% Missing
admission_date      : OK
bill_amount         : 10.0% Missing
department          : OK


### Three Strategies for Handling Missing Data

There is no single "correct" way to handle missing values. The right approach depends on **how much** data is missing, **why** it is missing, and **what** you plan to do with the data.

| Strategy | Method | When to Use |
|----------|--------|-------------|
| **Drop** | `.dropna()` | Small % missing, data is missing randomly, dataset is large enough |
| **Fill** | `.fillna()` | Need to preserve all rows, have a reasonable replacement value |
| **Flag** | Create indicator column | Missingness itself is informative (e.g., unanswered survey question) |

### Strategy 1: Drop rows with missing values — `.dropna()`

The simplest approach: remove any row that has a missing value. This is appropriate when:
- Only a small percentage of rows are affected
- The data is missing **randomly** (not systematically)
- Your dataset is large enough that losing some rows does not bias the results

In [8]:
# Strategy 1a: Drop ALL rows that have ANY missing value
patients_dropped_all = patients.dropna()
print(f"Before: {len(patients)} rows -> After: {len(patients_dropped_all)} rows")
print(f"Lost {len(patients) - len(patients_dropped_all)} rows ({(len(patients) - len(patients_dropped_all))/len(patients)*100:.0f}% of data)")

Before: 10 rows -> After: 6 rows
Lost 4 rows (40% of data)


In [9]:
# Strategy 1b: Drop rows only when SPECIFIC columns are missing
# More targeted -  we only care about missing age values
patients_dropped_age = patients.dropna(subset='age')
print(f"Before: {len(patients)} rows -> After {len(patients_dropped_age)} rows")
print(f"Only dropped rows where 'age' was missing")

Before: 10 rows -> After 8 rows
Only dropped rows where 'age' was missing


In [10]:
# We can also drop based on a threshold: keep rows with at least N non-null values
patients_thresh = patients.dropna(thresh=6)  # Keep rows with at least 6 non-null values
print(f"Before: {len(patients)} rows -> After: {len(patients_thresh)} rows")
print(f"Kept rows with at least 6 non-null values out of {len(patients.columns)} columns")

Before: 10 rows -> After: 9 rows
Kept rows with at least 6 non-null values out of 7 columns


### Strategy 2: Fill missing values — `.fillna()`

Instead of dropping rows, you can **replace** missing values with a substitute. Common fill strategies:

- **Constant value:** Fill with a known placeholder (e.g., `'Unknown'` for text, `0` for counts)
- **Statistical value:** Fill with the column's mean, median, or mode (for numeric data)
- **Forward/backward fill:** Use the previous or next row's value (for time series data)

In [11]:
# Strategy 2a: Fill with a constant value
patients_filled = patients.copy()
patients_filled['blood_group'] = patients_filled['blood_group'].fillna('Unknown')
print("Filled missing blood_group with 'Unknown':")
print(f"  Before: {patients['blood_group'].isna().sum()} missing")
print(f"  After:  {patients_filled['blood_group'].isna().sum()} missing")

Filled missing blood_group with 'Unknown':
  Before: 1 missing
  After:  0 missing


In [12]:
# Strategy 2b: Fill numeric columns with the mean (common for continuous data)
# First, we need to handle the 'N/A' string before we can compute a mean
# For now, let's work with the 'age' column which has proper NaN values
patients_filled_mean = patients.copy()
mean_age = patients_filled_mean['age'].mean()
patients_filled_mean['age'] = patients_filled_mean['age'].fillna(mean_age)
print(f"Mean age: {mean_age:.1f}")
print(f"Filled missing ages with mean value")
print(f"  Before: {patients['age'].isna().sum()} missing -> After: {patients_filled_mean['age'].isna().sum()} missing")

Mean age: 37.2
Filled missing ages with mean value
  Before: 2 missing -> After: 0 missing


In [13]:
# Strategy 2c: Forward fill (ffill) — use the PREVIOUS row's value
# Useful for time series data where values change slowly
patients_ffill = patients.copy()
patients_ffill['age'] = patients_ffill['age'].ffill()
print("Forward fill for 'age':")
print(f"  Row 2 (was None) -> now {patients_ffill.loc[2, 'age']} (copied from row 1)")
print(f"  Row 6 (was None) -> now {patients_ffill.loc[6, 'age']} (copied from row 5)")

Forward fill for 'age':
  Row 2 (was None) -> now 32.0 (copied from row 1)
  Row 6 (was None) -> now 41.0 (copied from row 5)


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — When NOT to Fill:</strong><br>
    Sometimes missing data IS the information. If a patient's blood_group is missing, filling it with 'A+' (the most common) could cause a <strong>fatal transfusion error</strong>. The right approach depends on the domain: fill numeric columns with mean/median for analysis, but leave critical medical fields as 'Unknown' or NaN.
</div>

### Strategy 3: The Hidden 'N/A' String Problem

This is one of the most common traps in real-world data. When someone enters `'N/A'` or `'null'` or `'-'` into a spreadsheet, it **looks** like a missing value to a human. But to Pandas, it is just a regular string — not missing at all.

Our `bill_amount` column has this exact problem: one value is `None` (actual NaN) and another is the string `'N/A'` (looks missing, but Pandas treats it as present text).

In [14]:
# The hidden problem: 'N/A' is NOT recognized as NaN
print(f"bill_amount dtype: {patients['bill_amount'].dtype}")  # object — should be numeric!
print(f"bill_amount values: {patients['bill_amount'].tolist()}")
print()
print("Notice: 'N/A' at index 5 is a STRING, not NaN")
print(f"pd.isna('N/A') = {pd.isna('N/A')}  <- Pandas does NOT treat 'N/A' string as missing!")

bill_amount dtype: object
bill_amount values: [15000, 22500, None, 18500, 20000.5, 'N/A', 17999.99, 19500, 22500, 16000]

Notice: 'N/A' at index 5 is a STRING, not NaN
pd.isna('N/A') = False  <- Pandas does NOT treat 'N/A' string as missing!


In [15]:
# Fix: Replace 'N/A' with actual NaN, then all missing values will be detectable.
patients_fixed = patients.copy()
patients_fixed['bill_amount'] = pd.to_numeric(patients_fixed['bill_amount'].replace('N/A', np.nan), errors = 'coerce')
print(f"Before fix: {patients['bill_amount'].isna().sum()} NaN detected")
print(f"After fix:  {patients_fixed['bill_amount'].isna().sum()} NaN detected")
print()
print("Now Pandas sees BOTH missing values in bill_amount!")
print(patients_fixed.dtypes)

Before fix: 1 NaN detected
After fix:  2 NaN detected

Now Pandas sees BOTH missing values in bill_amount!
patient_id            str
name                  str
age               float64
blood_group           str
admission_date        str
bill_amount       float64
department            str
dtype: object


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> Always check for string representations of missing data. Common culprits include <code>'N/A'</code>, <code>'NA'</code>, <code>'n/a'</code>, <code>'null'</code>, <code>'None'</code>, <code>'-'</code>, <code>''</code> (empty string), and <code>'missing'</code>. A quick way to check: <code>df['column'].unique()</code> — scan the output for any suspicious strings.
</div>

## Part 2: Data Type Conversion

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Currency Conversion:</strong><br>
    Think of data types like currencies. The number '100' means different things as dollars, rupees, or yen — the raw digits are identical, but the interpretation changes everything. Similarly, <code>'15000'</code> as a string is just five characters you can concatenate and search. As a number, it is a value you can add, compare, and average. Same raw data, different type, completely different behavior.
</div>

In [16]:
patients.dtypes

patient_id            str
name                  str
age               float64
blood_group           str
admission_date        str
bill_amount        object
department            str
dtype: object

Notice that `bill_amount` is `object` (string) type — it should be numeric. And `admission_date` is also `object` — it should be datetime. We cannot do math on strings or date arithmetic on text!

In [17]:
# The problem: can't do math on string columns
try:
    avg_bill = patients['bill_amount'].mean()
    print(f"Average bill: {avg_bill}")
except TypeError as e:
    print(f"TypeError: {e}")
    print()
    print("bill_amount is stored as 'object' (string) — Pandas can't compute mean on strings!")
    print(f"Actual values: {patients['bill_amount'].tolist()}")
    print("The culprit: 'N/A' at index 5 forced the entire column to be stored as text")

TypeError: unsupported operand type(s) for +: 'float' and 'str'

bill_amount is stored as 'object' (string) — Pandas can't compute mean on strings!
Actual values: [15000, 22500, None, 18500, 20000.5, 'N/A', 17999.99, 19500, 22500, 16000]
The culprit: 'N/A' at index 5 forced the entire column to be stored as text


In [18]:
# The naive fix FAILS: .astype(float) can't handle 'N/A' or None
try:
    patients['bill_amount'].astype(float)
except (ValueError, TypeError) as e:
    print(f"Error: {e}")
    print()
    print("You can't directly convert a column containing 'N/A' strings to float!")

Error: could not convert string to float: 'N/A'

You can't directly convert a column containing 'N/A' strings to float!


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — Two-Step Pattern: Clean FIRST, Convert SECOND.</strong><br>
    If you try <code>.astype(float)</code> on a column containing 'N/A' or other non-numeric strings, you get a ValueError. The solution: (1) Replace problematic strings with NaN, (2) THEN convert the type.
</div>

In [19]:
# Step 1: Replace 'N/A' string with actual NaN
patients_clean = patients.copy()
patients_clean['bill_amount'] = patients_clean['bill_amount'].replace('N/A', np.nan)
print("Step 1 done: Replaced 'N/A' with NaN")
print(f"Values: {patients_clean['bill_amount'].tolist()}")

Step 1 done: Replaced 'N/A' with NaN
Values: [15000, 22500, None, 18500, 20000.5, nan, 17999.99, 19500, 22500, 16000]


In [20]:
# Step 2: Safe conversion with pd.to_numeric(errors='coerce')
# 'coerce' means: if a value can't be converted, make it NaN instead of crashing
patients_clean['bill_amount'] = pd.to_numeric(patients_clean['bill_amount'], errors='coerce')
print(f"bill_amount dtype: {patients_clean['bill_amount'].dtype}")
print(f"Mean bill: Rs. {patients_clean['bill_amount'].mean():,.2f}")
print(f"Max bill:  Rs. {patients_clean['bill_amount'].max():,.2f}")
print()
print("Now we can do math!")

bill_amount dtype: float64
Mean bill: Rs. 19,000.06
Max bill:  Rs. 22,500.00

Now we can do math!


### Date Conversion — `pd.to_datetime()`

Our `admission_date` column has a tricky problem: **two different date formats** in the same column.

- Most rows use ISO format: `'2024-01-15'` (YYYY-MM-DD)
- One row uses a different format: `'15-01-2024'` (DD-MM-YYYY)

This is extremely common when data comes from multiple sources or is manually entered.

In [21]:
# The date problem: two different formats in the same column
print("Current admission_date values:")
for i, date in enumerate(patients['admission_date']):
    fmt = "DD-MM-YYYY" if date.startswith(('0','1','2','3')) and date[2] == '-' else "YYYY-MM-DD"
    print(f"  Row {i}: '{date}'  <- {fmt}")

Current admission_date values:
  Row 0: '2024-01-15'  <- YYYY-MM-DD
  Row 1: '2024-01-16'  <- YYYY-MM-DD
  Row 2: '2024-01-16'  <- YYYY-MM-DD
  Row 3: '15-01-2024'  <- DD-MM-YYYY
  Row 4: '2024-01-17'  <- YYYY-MM-DD
  Row 5: '2024-01-18'  <- YYYY-MM-DD
  Row 6: '2024-01-19'  <- YYYY-MM-DD
  Row 7: '2024-01-20'  <- YYYY-MM-DD
  Row 8: '2024-01-20'  <- YYYY-MM-DD
  Row 9: '2024-01-21'  <- YYYY-MM-DD


In [22]:
# pd.to_datetime() with format='mixed' handles multiple formats
patients_clean['admission_date'] = pd.to_datetime(
    patients['admission_date'],
    dayfirst=True, # Tell Pandas that day comes first in ambiguous dates
    format='mixed' # Allow mixed formats in the same column
)
print(f"admission_date dtype: {patients_clean['admission_date'].dtype}")
print()
print("Converted dates:")
for i, date in enumerate(patients_clean['admission_date']):
    print(f"  Row {i}: {date.strftime('%Y-%m-%d')}")

admission_date dtype: datetime64[us]

Converted dates:
  Row 0: 2024-01-15
  Row 1: 2024-01-16
  Row 2: 2024-01-16
  Row 3: 2024-01-15
  Row 4: 2024-01-17
  Row 5: 2024-01-18
  Row 6: 2024-01-19
  Row 7: 2024-01-20
  Row 8: 2024-01-20
  Row 9: 2024-01-21


In [23]:
# Now we can do date operations!
print(f"Date range: {patients_clean['admission_date'].min().strftime('%Y-%m-%d')} to {patients_clean['admission_date'].max().strftime('%Y-%m-%d')}")
print(f"Span: {(patients_clean['admission_date'].max() - patients_clean['admission_date'].min()).days} days")

Date range: 2024-01-15 to 2024-01-21
Span: 6 days


<div style="background: #EBF5FB; color: gray; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Key Concept — The dtype Hierarchy:</strong><br><br>
    <code>object</code> → <code>numeric</code> / <code>datetime</code> → <code>categorical</code><br><br>
    When you load a CSV, Pandas guesses data types. If ANY value in a column is non-numeric (like 'N/A'), the ENTIRE column becomes <code>object</code> (string). This is why <code>.dtypes</code> is always your first diagnostic check — if a numeric column shows as <code>object</code>, something is wrong.
</div>

## Part 3: String Operations - The .str Accessor

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Find-and-Replace for Entire Columns:</strong><br>
    Think of the <code>.str</code> accessor as <strong>Find-and-Replace in a word processor</strong> — but for an entire column at once. Instead of fixing text one cell at a time (the Excel way), Pandas lets you apply a transformation to every value in a column with one line of code.
</div>

In [95]:
# The problem: inconsistent casing makes analysis WRONG
# Pandas treats 'Cardiology', 'cardiology', and 'CARDIOLOGY' as THREE different departments!
patients['department'].value_counts()

department
Cardiology     3
cardiology     3
Orthopedics    2
orthopedics    1
CARDIOLOGY     1
Name: count, dtype: int64

### The `.str` Accessor

Any string method you know from Python (`lower()`, `upper()`, `strip()`, `replace()`, `contains()`) can be applied to an **entire column** by putting `.str.` before it.

| Python String Method | Pandas Column Equivalent | What It Does |
|---------------------|-------------------------|-------------|
| `"hello".lower()` | `df['col'].str.lower()` | Convert to lowercase |
| `"hello".upper()` | `df['col'].str.upper()` | Convert to UPPERCASE |
| `"hello".title()` | `df['col'].str.title()` | Convert to Title Case |
| `"hello".strip()` | `df['col'].str.strip()` | Remove leading/trailing whitespace |
| `"lo" in "hello"` | `df['col'].str.contains("lo")` | Check if substring exists |

In [96]:
# .str.lower() — convert all values to lowercase
patients['department'].str.lower()

0     cardiology
1     cardiology
2    orthopedics
3    orthopedics
4     cardiology
5     cardiology
6    orthopedics
7     cardiology
8     cardiology
9     cardiology
Name: department, dtype: str

In [97]:
# .str.upper() — convert all values to UPPERCASE
patients['department'].str.upper()

0     CARDIOLOGY
1     CARDIOLOGY
2    ORTHOPEDICS
3    ORTHOPEDICS
4     CARDIOLOGY
5     CARDIOLOGY
6    ORTHOPEDICS
7     CARDIOLOGY
8     CARDIOLOGY
9     CARDIOLOGY
Name: department, dtype: str

In [98]:
# .str.title() — convert to Title Case (capitalize first letter of each word)
# This is the preferred format for proper nouns and department names
patients['department'].str.title()

0     Cardiology
1     Cardiology
2    Orthopedics
3    Orthopedics
4     Cardiology
5     Cardiology
6    Orthopedics
7     Cardiology
8     Cardiology
9     Cardiology
Name: department, dtype: str

In [99]:
# .str.strip() — remove leading and trailing whitespace
# Look at ' Priya Sharma ' — it has spaces on BOTH sides
print(f"Before strip: '{patients.loc[1, 'name']}'  (length: {len(patients.loc[1, 'name'])})")
print(f"After strip:  '{patients.loc[1, 'name'].strip()}'  (length: {len(patients.loc[1, 'name'].strip())})")

Before strip: ' Priya Sharma '  (length: 14)
After strip:  'Priya Sharma'  (length: 12)


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — Always strip BEFORE casing!</strong><br>
    <code>' Priya Sharma '.title()</code> produces <code>' Priya Sharma '</code> — the leading space REMAINS and causes merge/GroupBy problems. The correct order: <code>.str.strip().str.title()</code> — strip whitespace first, THEN standardize case.
</div>

In [100]:
# The correct pattern: strip → then case standardize
# Apply to the 'name' and 'department' columns
patients_str_clean = patients.copy()
patients_str_clean['name'] = patients_str_clean['name'].str.strip().str.title()
patients_str_clean['department'] = patients_str_clean['department'].str.strip().str.title()

print("Before cleaning:")
print(f"  Unique departments: {patients['department'].nunique()} → {patients['department'].unique().tolist()}")
print()
print("After cleaning:")
print(f"  Unique departments: {patients_str_clean['department'].nunique()} → {patients_str_clean['department'].unique().tolist()}")

Before cleaning:
  Unique departments: 5 → ['Cardiology', 'cardiology', 'Orthopedics', 'orthopedics', 'CARDIOLOGY']

After cleaning:
  Unique departments: 2 → ['Cardiology', 'Orthopedics']


In [101]:
# .str.contains() — search for a substring within each value
# Find all patients in Cardiology-related departments (case-insensitive)
cardio_mask = patients['department'].str.contains('cardio', case=False, na=False)
print(f"Cardiology patients: {cardio_mask.sum()} out of {len(patients)}")

Cardiology patients: 7 out of 10


In [102]:
# Display the Cardiology patients
patients[cardio_mask][['patient_id', 'name', 'department']]

,patient_id,name,department
0,P001,Rajesh Kumar,Cardiology
1,P002,Priya Sharma,cardiology
4,P005,Sneha Iyer,Cardiology
5,P006,NaN,CARDIOLOGY
7,P008,karan singh,cardiology
8,P002,Priya Sharma,cardiology
9,P010,Divya Rao,Cardiology


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice — The Complete String Cleaning Chain:</strong><br>
    <code>.str.strip()</code> (remove whitespace) → <code>.str.title()</code> or <code>.str.lower()</code> (standardize case) → <code>.str.replace()</code> (fix specific values). Always apply in this order for consistent results.
</div>

## Part 4: Handling Duplicates

<div style="background: #F5F5F5; color: black; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Hospital Wristband:</strong><br>
    Imagine a patient arrives at the emergency department. Due to a system glitch, two wristbands are printed with the same patient ID. Now: The nurse scans wristband 1 and records vitals. A different nurse scans wristband 2 and records vitals AGAIN. The billing system counts TWO admissions — the patient is billed twice. Duplicate records don't just waste storage — they corrupt every aggregation, average, and count you compute.
</div>

In [103]:
# Detect duplicates: .duplicated() returns True for duplicate records
patients.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool

In [104]:
# How many duplicates?
print(f"Number of duplicate rows: {patients.duplicated().sum()}")

Number of duplicate rows: 0


In [105]:
# Show the duplicate rows — which ones are copies?
patients[patients.duplicated(keep=False)]  # keep=False marks ALL copies, not just the second one

,patient_id,name,age,blood_group,admission_date,bill_amount,department


### Understanding the `keep` Parameter

The `keep` parameter controls which occurrence is marked as a duplicate:

- **`keep='first'`** (default): Marks the **second** (and subsequent) occurrence as duplicate. The first occurrence is kept.
- **`keep='last'`**: Marks the **first** occurrence as duplicate. The last occurrence is kept.
- **`keep=False`**: Marks **ALL** occurrences as duplicates — useful for inspection to see every copy.

In [106]:
# Compare keep options
print("keep='first' (default):", patients.duplicated(keep='first').sum(), "duplicates")
print("keep='last':           ", patients.duplicated(keep='last').sum(), "duplicates")
print("keep=False:            ", patients.duplicated(keep=False).sum(), "duplicates (marks ALL copies)")

keep='first' (default): 0 duplicates
keep='last':            0 duplicates
keep=False:             0 duplicates (marks ALL copies)


In [107]:
# Check duplicates on a SPECIFIC column (more useful in practice)
# Two patients with the same patient_id should NOT exist
patients.duplicated(subset=['patient_id'])

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8     True
9    False
dtype: bool

In [108]:
# Show which patient_ids are duplicated
print("Duplicated patient IDs:")
dup_ids = patients[patients.duplicated(subset=['patient_id'], keep=False)]
print(f"Found {len(dup_ids)} rows with duplicate patient_id")

Duplicated patient IDs:
Found 2 rows with duplicate patient_id


In [109]:
dup_ids[['patient_id', 'name', 'age', 'department']]

,patient_id,name,age,department
1,P002,Priya Sharma,32.0,cardiology
8,P002,Priya Sharma,32.0,cardiology


In [110]:
# Remove duplicates — keep the FIRST occurrence
patients_deduped = patients.drop_duplicates(subset=['patient_id'], keep='first')
print(f"Before: {len(patients)} rows -> After: {len(patients_deduped)} rows")
print(f"Removed {len(patients) - len(patients_deduped)} duplicate(s)")

Before: 10 rows -> After: 9 rows
Removed 1 duplicate(s)


<div style="background: #EBF5FB; color: gray; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Key Concept — Exact vs Partial Duplicates:</strong><br>
    <strong>Exact duplicates</strong> are rows where EVERY column matches (use <code>.duplicated()</code> with no <code>subset</code>). <strong>Partial duplicates</strong> share the same key column (e.g., patient_id) but may differ in other fields — these are trickier and require domain knowledge to resolve. In our example, P002 is an exact duplicate — both rows are identical.
</div>

<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> Always check for duplicates BEFORE and AFTER merging. Merging two tables can CREATE duplicates if the join key has a many-to-many relationship. A quick sanity check: compare <code>len(df)</code> before and after the merge.
</div>

## Part 5: Tidy Data Principles

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Filing Cabinet vs Loose Papers:</strong><br>
    Think of tidy data like a well-organized <strong>filing cabinet</strong> vs a drawer full of loose papers. Filing cabinet: each drawer is labeled (column), each folder inside is one person's file (row), and you have separate cabinets for HR, Finance, Sales (separate tables). Loose-paper drawer: everything is mixed together — you have to dig through the whole pile every time you need something.
</div>

<div style="background: #EBF5FB; color: gray; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>The Three Rules of Tidy Data (Hadley Wickham, 2014):</strong><br><br>
    1. <strong>Each variable</strong> forms its own <strong>column</strong><br>
    2. <strong>Each observation</strong> forms its own <strong>row</strong><br>
    3. <strong>Each value</strong> occupies its own <strong>cell</strong><br><br>
    If your data follows these rules, every Pandas operation (filter, group, merge, plot) becomes straightforward. If it doesn't, you need to <strong>reshape</strong> it first.
</div>

In [111]:
# Example of UNTIDY (wide) data — months are stored as columns
sales_wide = pd.DataFrame({
    'salesperson': ['Priya', 'Rahul', 'Anita'],
    'Jan': [45000, 38000, 52000],
    'Feb': [48000, 41000, 49000],
    'Mar': [51000, 43000, 55000]
})

print("UNTIDY (wide format) — months are column names, not values:")

UNTIDY (wide format) — months are column names, not values:


In [112]:
sales_wide

,salesperson,Jan,Feb,Mar
0,Priya,45000,48000,51000
1,Rahul,38000,41000,43000
2,Anita,52000,49000,55000


In [113]:
# pd.melt() — convert wide format to tidy (long) format
# id_vars: columns to keep as-is
# var_name: name for the column created from old column headers
# value_name: name for the column created from old values
sales_tidy = pd.melt(
    sales_wide,
    id_vars=['salesperson'],
    var_name='month',
    value_name='sales'
)

print(f"Wide: {sales_wide.shape} → Tidy: {sales_tidy.shape}")
print("Now each row is ONE observation: one salesperson's sales in one month")

Wide: (3, 4) → Tidy: (9, 3)
Now each row is ONE observation: one salesperson's sales in one month


In [114]:
sales_tidy

,salesperson,month,sales
0,Priya,Jan,45000
1,Rahul,Jan,38000
2,Anita,Jan,52000
3,Priya,Feb,48000
4,Rahul,Feb,41000
5,Anita,Feb,49000
6,Priya,Mar,51000
7,Rahul,Mar,43000
8,Anita,Mar,55000


In [115]:
# pivot_table() — convert tidy (long) back to wide format
# Useful for presentation/reporting
sales_back_to_wide = sales_tidy.pivot_table(
    values='sales',
    index='salesperson',
    columns='month'
)

print("Back to wide format (for presentation):")

Back to wide format (for presentation):


In [116]:
sales_back_to_wide

month,Feb,Jan,Mar
salesperson,,,
Anita,49000.0,52000.0,55000.0
Priya,48000.0,45000.0,51000.0
Rahul,41000.0,38000.0,43000.0


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> <strong>Tidy for analysis, wide for presentation.</strong> Use <code>pd.melt()</code> to reshape wide data into tidy format before analysis (filtering, grouping, plotting). Use <code>pd.pivot_table()</code> to reshape tidy data back to wide for final reports and dashboards.
</div>

## Part 6: Combining DataFrames - pd.concat()

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Cafeteria Trays:</strong><br>
    Think of <code>pd.concat()</code> like stacking <strong>cafeteria trays</strong>. <strong>Vertical stacking (axis=0):</strong> Putting one tray on top of another — more food items (rows), same categories (columns). <strong>Horizontal stacking (axis=1):</strong> Putting trays side by side — same items (rows), more information about each (columns).
</div>

In [117]:
# January orders
jan_orders = pd.DataFrame({
    'order_id': ['ORD001', 'ORD002', 'ORD003'],
    'product': ['Mouse', 'Keyboard', 'Monitor'],
    'amount': [899, 3499, 8999]
})

# February orders
feb_orders = pd.DataFrame({
    'order_id': ['ORD004', 'ORD005', 'ORD006'],
    'product': ['Headphones', 'USB Hub', 'Webcam'],
    'amount': [1999, 1299, 2499]
})

print("January orders:")
jan_orders

January orders:


,order_id,product,amount
0,ORD001,Mouse,899
1,ORD002,Keyboard,3499
2,ORD003,Monitor,8999


In [118]:
print("February Orders:")
feb_orders

February Orders:


,order_id,product,amount
0,ORD004,Headphones,1999
1,ORD005,USB Hub,1299
2,ORD006,Webcam,2499


In [119]:
# Vertical stack (axis=0) — combine months into one table
all_orders = pd.concat([jan_orders, feb_orders])
print(f"Jan: {len(jan_orders)} rows + Feb: {len(feb_orders)} rows = Combined: {len(all_orders)} rows")
print()
print("Notice the INDEX problem — indices 0,1,2 appear twice!")

Jan: 3 rows + Feb: 3 rows = Combined: 6 rows

Notice the INDEX problem — indices 0,1,2 appear twice!


In [120]:
all_orders

,order_id,product,amount
0,ORD001,Mouse,899
1,ORD002,Keyboard,3499
2,ORD003,Monitor,8999
0,ORD004,Headphones,1999
1,ORD005,USB Hub,1299
2,ORD006,Webcam,2499


In [121]:
# Fix: ignore_index=True creates a clean sequential index
all_orders_clean = pd.concat([jan_orders, feb_orders], ignore_index=True)
all_orders_clean

,order_id,product,amount
0,ORD001,Mouse,899
1,ORD002,Keyboard,3499
2,ORD003,Monitor,8999
3,ORD004,Headphones,1999
4,ORD005,USB Hub,1299
5,ORD006,Webcam,2499


In [122]:
# Horizontal stack (axis=1) — add more columns side by side
# Customer info for the same orders
customer_info = pd.DataFrame({
    'customer': ['Priya', 'Rahul', 'Anita'],
    'city': ['Mumbai', 'Delhi', 'Bangalore']
})

# Side-by-side combination
combined_horizontal = pd.concat([jan_orders, customer_info], axis=1)
combined_horizontal

,order_id,product,amount,customer,city
0,ORD001,Mouse,899,Priya,Mumbai
1,ORD002,Keyboard,3499,Rahul,Delhi
2,ORD003,Monitor,8999,Anita,Bangalore


<div style="background: #FEF9E7; color: gray; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning:</strong> <code>pd.concat()</code> does NOT match rows intelligently. It simply glues DataFrames together by position (axis=0: stack vertically, axis=1: stack horizontally). If you need to match rows based on a shared key column (like customer_id), use <code>pd.merge()</code> instead.
</div>

<div style="background: #EBF5FB; color: black; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>concat vs merge — When to Use Which:</strong><br><br>
    <code>concat</code> = <strong>glue</strong> (just stick things together by position).<br>
    <code>merge</code> = <strong>match-and-combine</strong> (find matching keys, then combine).<br><br>
    Use <strong>concat</strong> for appending similar datasets (e.g., monthly reports).<br>
    Use <strong>merge</strong> for combining different datasets that share a key column.
</div>

## Part 7: Merging DataFrames - pd.merge()

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — VLOOKUP on Steroids:</strong><br>
    If you have used Excel, <code>pd.merge()</code> is like VLOOKUP — but faster, more powerful, and it can handle millions of rows without crashing. If you haven't used Excel: imagine you have a school <strong>attendance register</strong> (student_id, date, present/absent) and a separate <strong>student directory</strong> (student_id, name, phone, address). Merge is how you combine them to see WHO was absent, not just which ID.
</div>

### The Four Join Types

| Join Type | What It Keeps | Analogy |
|-----------|--------------|--------|
| **inner** | Only rows with matching keys in BOTH tables | The intersection of two circles |
| **left** | ALL rows from left + matches from right (NaN if no match) | Keep everything from my table, add info if available |
| **right** | ALL rows from right + matches from left (NaN if no match) | Keep everything from the other table |
| **outer** | ALL rows from BOTH tables (NaN where no match) | The union of two circles |

In [123]:
# Student grades from Math class
math_grades = pd.DataFrame({
    'student_id': ['S001', 'S002', 'S003', 'S004'],
    'name': ['Priya', 'Rahul', 'Anita', 'Vikram'],
    'math_score': [85, 72, 91, 68]
})

# Student grades from Science class (S003 didn't take Science, S005 is new)
science_grades = pd.DataFrame({
    'student_id': ['S001', 'S002', 'S004', 'S005'],
    'science_score': [78, 88, 65, 92]
})

print("Math class: 4 students (S001, S002, S003, S004)")
print("Science class: 4 students (S001, S002, S004, S005)")
print("Note: S003 (Anita) didn't take Science, S005 is a new student")

Math class: 4 students (S001, S002, S003, S004)
Science class: 4 students (S001, S002, S004, S005)
Note: S003 (Anita) didn't take Science, S005 is a new student


In [124]:
math_grades

,student_id,name,math_score
0,S001,Priya,85
1,S002,Rahul,72
2,S003,Anita,91
3,S004,Vikram,68


In [125]:
science_grades

,student_id,science_score
0,S001,78
1,S002,88
2,S004,65
3,S005,92


In [126]:
# INNER JOIN: only students who took BOTH subjects
inner_result = pd.merge(math_grades, science_grades, on='student_id', how='inner')
print(f"Inner join: {len(math_grades)} left + {len(science_grades)} right → {len(inner_result)} rows")
print("Only students present in BOTH tables survive")

Inner join: 4 left + 4 right → 3 rows
Only students present in BOTH tables survive


In [127]:
inner_result

,student_id,name,math_score,science_score
0,S001,Priya,85,78
1,S002,Rahul,72,88
2,S004,Vikram,68,65


In [128]:
# LEFT JOIN: keep ALL students from math, add science IF available
left_result = pd.merge(math_grades, science_grades, on='student_id', how='left')
print(f"Left join: {len(math_grades)} left + {len(science_grades)} right → {len(left_result)} rows")
print("All math students kept; S003 gets NaN for science_score")

Left join: 4 left + 4 right → 4 rows
All math students kept; S003 gets NaN for science_score


In [129]:
left_result

,student_id,name,math_score,science_score
0,S001,Priya,85,78.0
1,S002,Rahul,72,88.0
2,S003,Anita,91,NaN
3,S004,Vikram,68,65.0


In [130]:
# RIGHT JOIN: keep ALL students from science, add math IF available
right_result = pd.merge(math_grades, science_grades, on='student_id', how='right')
print(f"Right join: {len(math_grades)} left + {len(science_grades)} right → {len(right_result)} rows")
print("All science students kept; S005 gets NaN for name and math_score")

Right join: 4 left + 4 right → 4 rows
All science students kept; S005 gets NaN for name and math_score


In [131]:
right_result

,student_id,name,math_score,science_score
0,S001,Priya,85.0,78
1,S002,Rahul,72.0,88
2,S004,Vikram,68.0,65
3,S005,NaN,NaN,92


In [132]:
# OUTER JOIN: keep EVERYONE from BOTH tables
outer_result = pd.merge(math_grades, science_grades, on='student_id', how='outer')
print(f"Outer join: {len(math_grades)} left + {len(science_grades)} right → {len(outer_result)} rows")
print("Nobody is lost — NaN fills in the gaps")

Outer join: 4 left + 4 right → 5 rows
Nobody is lost — NaN fills in the gaps


In [133]:
outer_result

,student_id,name,math_score,science_score
0,S001,Priya,85.0,78.0
1,S002,Rahul,72.0,88.0
2,S003,Anita,91.0,NaN
3,S004,Vikram,68.0,65.0
4,S005,NaN,NaN,92.0


<div style="background: #EBF5FB; color: black; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Join Type Quick Reference:</strong><br><br>
    • <code>inner</code>: Only the overlap (A AND B)<br>
    • <code>left</code>: Everything from A, matches from B<br>
    • <code>right</code>: Everything from B, matches from A<br>
    • <code>outer</code>: Everything from A AND B<br><br>
    <strong>Default is <code>inner</code>.</strong> In practice, <code>left</code> is the most common — you usually want to keep all rows from your primary table and enrich with additional info.
</div>

### Merge Parameters Deep Dive

In [134]:
# When the key columns have DIFFERENT names in each DataFrame
employees = pd.DataFrame({
    'emp_id': ['E001', 'E002', 'E003'],
    'name': ['Priya', 'Rahul', 'Anita']
})

salaries = pd.DataFrame({
    'employee_id': ['E001', 'E002', 'E003'],
    'salary': [75000, 62000, 88000]
})

# Use left_on and right_on when column names differ
merged = pd.merge(employees, salaries, left_on='emp_id', right_on='employee_id')
merged

,emp_id,name,employee_id,salary
0,E001,Priya,E001,75000
1,E002,Rahul,E002,62000
2,E003,Anita,E003,88000


In [135]:
# indicator=True adds a '_merge' column showing where each row came from
indicator_result = pd.merge(
    math_grades, science_grades,
    on='student_id',
    how='outer',
    indicator=True
)
indicator_result

,student_id,name,math_score,science_score,_merge
0,S001,Priya,85.0,78.0,both
1,S002,Rahul,72.0,88.0,both
2,S003,Anita,91.0,NaN,left_only
3,S004,Vikram,68.0,65.0,both
4,S005,NaN,NaN,92.0,right_only


<div style="background: #FEF9E7; color: gray; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Merge Sanity Checks — Always verify after merging:</strong><br><br>
    1. <strong>Column count:</strong> result should have <code>left_cols + right_cols - shared_keys</code> columns<br>
    2. <strong>Row count:</strong> for a left join, result rows should equal left table rows (if key is unique in right table)<br>
    3. <strong>NaN check:</strong> <code>result.isna().sum()</code> — understand where gaps come from<br><br>
    If your row count INCREASES unexpectedly, you likely have duplicate keys causing a many-to-many join!
</div>

In [136]:
# Sanity check pattern — ALWAYS do this after a merge
print("=== Merge Sanity Check ===")
print(f"Left table:   {math_grades.shape}")
print(f"Right table:  {science_grades.shape}")
print(f"Result table: {left_result.shape}")
print(f"Expected cols: {math_grades.shape[1] + science_grades.shape[1] - 1} (shared key: student_id)")
print(f"Actual cols:   {left_result.shape[1]}")
print(f"NaN values:")
print(left_result.isna().sum())

=== Merge Sanity Check ===
Left table:   (4, 3)
Right table:  (4, 2)
Result table: (4, 4)
Expected cols: 4 (shared key: student_id)
Actual cols:   4
NaN values:
student_id       0
name             0
math_score       0
science_score    1
dtype: int64


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> <strong>Left join is the safer default in practice</strong> — you never want to silently lose rows from your primary dataset. Use inner join only when you explicitly want to keep only matched records. Always check row counts before and after.
</div>

## Part 8: GroupBy -> Split - Apply - Combine

<div style="background: #F5F5F5; color: gray; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Sorting Exam Papers:</strong><br>
    Imagine you are a teacher with 200 exam papers from 5 different sections. <strong>Split:</strong> Sort the papers into 5 piles — one pile per section. <strong>Apply:</strong> Calculate the average score for each pile. <strong>Combine:</strong> Write the 5 averages on the board as a summary table. You never mixed the piles. You calculated WITHIN each pile. Then you combined the results. That's exactly what <code>.groupby()</code> does.
</div>

<div style="background: #EBF5FB; color: black; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>The Split-Apply-Combine Pattern:</strong>
<pre>
Original DataFrame
       ↓
    SPLIT by group key (e.g., department)
       ↓
    ┌──────────┐  ┌───────────┐  ┌─────────┐
    │Cardiology│  │Orthopedics│  │  ...    │
    │  pile    │  │   pile    │  │  pile   │
    └──────────┘  └───────────┘  └─────────┘
       ↓              ↓              ↓
    APPLY function (e.g., mean) to each pile
       ↓              ↓              ↓
    ┌──────┐      ┌──────┐      ┌──────┐
    │ 19500│      │ 18000│      │ ...  │
    └──────┘      └──────┘      └──────┘
       ↓
    COMBINE into summary table
</pre>
</div>

In [137]:
# Create a clean version of our patients data for GroupBy demos
# (applying everything we learned in Parts 1-4)
patients_gb = patients.copy()

# Clean bill_amount: replace 'N/A' → NaN, convert to numeric
patients_gb['bill_amount'] = patients_gb['bill_amount'].replace('N/A', np.nan)
patients_gb['bill_amount'] = pd.to_numeric(patients_gb['bill_amount'], errors='coerce')

# Clean department: standardize casing
patients_gb['department'] = patients_gb['department'].str.strip().str.title()

# Remove duplicate P002
patients_gb = patients_gb.drop_duplicates(subset=['patient_id'], keep='first')

print(f"Clean dataset: {patients_gb.shape[0]} rows × {patients_gb.shape[1]} columns")
print(f"Unique departments: {patients_gb['department'].unique().tolist()}")

Clean dataset: 9 rows × 7 columns
Unique departments: ['Cardiology', 'Orthopedics']


In [138]:
# Simple groupby: average bill amount per department
patients_gb.groupby('department')['bill_amount'].mean()

department
Cardiology     18600.100
Orthopedics    18249.995
Name: bill_amount, dtype: float64

In [139]:
patients_gb[patients_gb["department"]=="Cardiology"]["bill_amount"].mean()

np.float64(18600.1)

In [140]:
patients_gb[patients_gb["department"]=="Orthopedics"]["bill_amount"].mean()

np.float64(18249.995000000003)

In [141]:
# What does a GroupBy object look like?
grouped = patients_gb.groupby('department')
print(f"Type: {type(grouped)}")
print(f"Number of groups: {grouped.ngroups}")
print()
for name, group in grouped:
    print(f"Group '{name}': {len(group)} patients")

Type: <class 'pandas.api.typing.DataFrameGroupBy'>
Number of groups: 2

Group 'Cardiology': 6 patients
Group 'Orthopedics': 3 patients


In [142]:
# Multiple aggregations on one column
patients_gb.groupby('department')['bill_amount'].agg(['mean', 'sum', 'count', 'min', 'max'])

,mean,sum,count,min,max
department,,,,,
Cardiology,18600.100,93000.50,5,15000.00,22500.0
Orthopedics,18249.995,36499.99,2,17999.99,18500.0


In [143]:
# Named aggregations — more readable and professional
department_summary = patients_gb.groupby('department').agg(
    patient_count=('patient_id', 'count'),
    avg_bill=('bill_amount', 'mean'),
    total_revenue=('bill_amount', 'sum'),
    avg_age=('age', 'mean')
).reset_index()

department_summary

,department,patient_count,avg_bill,total_revenue,avg_age
0,Cardiology,6,18600.100,93000.50,39.666667
1,Orthopedics,3,18249.995,36499.99,28.000000


In [144]:
# Apply different functions to different columns
patients_gb.groupby('department').agg({
    'patient_id': 'count',        # count patients
    'bill_amount': ['mean', 'sum'],  # average and total bills
    'age': 'mean'                 # average age
})

patient_id bill_amount                  age
                 count        mean       sum       mean
department                                             
Cardiology           6   18600.100  93000.50  39.666667
Orthopedics          3   18249.995  36499.99  28.000000

### Pivot Tables — The Presentation-Ready GroupBy

A pivot table is essentially a GroupBy operation displayed in a cross-tabulation format — rows are one grouping variable, columns are another. This is the format managers and stakeholders actually want to see.

In [145]:
patients_gb

,patient_id,name,age,blood_group,admission_date,bill_amount,department
0,P001,Rajesh Kumar,45.0,A+,2024-01-15,15000.00,Cardiology
1,P002,Priya Sharma,32.0,B+,2024-01-16,22500.00,Cardiology
2,P003,ANITA DESAI,NaN,O-,2024-01-16,NaN,Orthopedics
3,P004,vikram patel,28.0,AB+,15-01-2024,18500.00,Orthopedics
4,P005,Sneha Iyer,56.0,A-,2024-01-17,20000.50,Cardiology
5,P006,NaN,41.0,B+,2024-01-18,NaN,Cardiology
6,P007,Meera Joshi,NaN,O+,2024-01-19,17999.99,Orthopedics
7,P008,karan singh,35.0,NaN,2024-01-20,19500.00,Cardiology
9,P010,Divya Rao,29.0,A+,2024-01-21,16000.00,Cardiology


In [146]:
# Pivot table: department summary
pivot = pd.pivot_table(
    patients_gb,
    values='bill_amount',
    index='department',
    aggfunc=['count', 'mean']
)

pivot

,count,mean
,bill_amount,bill_amount
department,,
Cardiology,5,18600.100
Orthopedics,2,18249.995


In [147]:
# Add margins (row/column totals) — makes it report-ready
pivot_with_totals = pd.pivot_table(
    patients_gb,
    values='bill_amount',
    index='department',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

pivot_with_totals

,bill_amount
department,
Cardiology,93000.50
Orthopedics,36499.99
Total,129500.49


<div style="background: #FEF9E7; color: gray; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning:</strong> <strong>GroupBy drops NaN groups by default.</strong> If a group key contains NaN values, that group is silently excluded from results. Use <code>dropna=False</code> in the groupby call to include NaN as its own group: <code>df.groupby('col', dropna=False).mean()</code>
</div>

In [148]:
# Demonstrate: GroupBy drops NaN groups by default
print("With dropna=True (default):")
print(f"  Groups: {patients_gb.groupby('blood_group').ngroups}")
print()
print("With dropna=False:")
print(f"  Groups: {patients_gb.groupby('blood_group', dropna=False).ngroups}")
print("  (NaN becomes its own group)")

With dropna=True (default):
  Groups: 6

With dropna=False:
  Groups: 7
  (NaN becomes its own group)


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> <strong>GroupBy + sort_values = instant ranking.</strong> The most common analytics pattern: group by a category, aggregate, then sort to find the top/bottom performers. Chain it in one expression: <code>df.groupby('category').agg(total=('amount', 'sum')).sort_values('total', ascending=False).head(3)</code>
</div>

In [149]:
# The Top N pattern: GroupBy → Aggregate → Sort → Head
# Which department generates the most revenue?
(patients_gb
 .groupby('department')
 .agg(total_revenue=('bill_amount', 'sum'))
 .sort_values('total_revenue', ascending=False)
)

,total_revenue
department,
Cardiology,93000.50
Orthopedics,36499.99


## Mini Project: Multi-Source Sales Report

You are a junior data analyst at an Indian e-commerce company. The sales team, customer team, and product team each maintain their own spreadsheet. Your manager needs a **unified sales report** broken down by city, category, and customer segment.

The catch: **all three datasets are messy.** Welcome to real-world data analysis.

Your pipeline:
1. **Create** the 3 messy datasets
2. **Explore** each dataset (find the problems)
3. **Clean** each dataset (fix the problems)
4. **Merge** them into a unified report
5. **Analyze** with GroupBy and Pivot Tables

### Step 1: Create Three Messy Datasets

In [150]:
# ========================================
# MINI-PROJECT DATASET 1: Orders (Sales Team's Spreadsheet)
# 20 orders + 1 deliberate duplicate = 21 rows
# Problems: null customer_ids, duplicate row, dates as strings
# ========================================

orders = pd.DataFrame({
    'order_id': [f'ORD{str(i).zfill(3)}' for i in range(1, 21)],
    'customer_id': ['C001','C002','C003','C004','C005','C001','C002','C003',None,'C005',
                    'C006','C007','C008','C009','C010','C006',None,'C008','C009','C010'],
    'product_id': ['P101','P102','P103','P104','P105','P106','P107','P108','P109','P110',
                   'P101','P102','P103','P104','P105','P106','P107','P108','P109','P110'],
    'quantity': [2, 1, 3, 1, 2, 1, 4, 2, 1, 3, 2, 1, 1, 2, 3, 1, 2, 1, 4, 2],
    'order_date': ['2024-01-15','2024-01-15','2024-01-16','2024-01-17','2024-01-17',
                   '2024-01-18','2024-01-18','2024-01-19','2024-01-20','2024-01-20',
                   '2024-01-21','2024-01-21','2024-01-22','2024-01-22','2024-01-23',
                   '2024-01-23','2024-01-24','2024-01-24','2024-01-25','2024-01-25']
})

# Add deliberate duplicate (row index 2 repeated)
orders = pd.concat([orders, orders.iloc[[2]]], ignore_index=True)

print(f"Orders: {orders.shape[0]} rows x {orders.shape[1]} columns")
print(f"(Suspicious: 21 rows for 20 order IDs...)")

Orders: 21 rows x 5 columns
(Suspicious: 21 rows for 20 order IDs...)


In [151]:
# ========================================
# MINI-PROJECT DATASET 2: Customers (Customer Team's Spreadsheet)
# Problems: inconsistent name/city casing, leading whitespace
# ========================================

customers = pd.DataFrame({
    'customer_id': ['C001','C002','C003','C004','C005','C006','C007','C008','C009','C010'],
    'name': ['Priya Sharma','Rahul Verma',' Anita Desai','VIKRAM PATEL','sneha iyer',
             'Arjun Nair','Meera Joshi','Karan Singh','Divya Rao','Amit Gupta'],
    'city': ['Mumbai','Delhi','Bangalore',' chennai','Mumbai','KOLKATA','pune','Delhi','bangalore','Mumbai'],
    'segment': ['Premium','Regular','Premium','Regular','Premium','Regular','Premium','Regular','Premium','Regular']
})

print(f"Customers: {customers.shape[0]} rows x {customers.shape[1]} columns")

Customers: 10 rows x 4 columns


In [152]:
# ========================================
# MINI-PROJECT DATASET 3: Products (Product Team's Spreadsheet)
# Problems: 1 missing category, price stored as 'Rs. XXX' strings
# ========================================

products = pd.DataFrame({
    'product_id': ['P101','P102','P103','P104','P105','P106','P107','P108','P109','P110'],
    'product_name': ['Wireless Mouse','Python Book','USB-C Hub','Cotton T-Shirt','Desk Lamp',
                     'Data Science Book','Bluetooth Speaker','Running Shoes','Notebook Set','Keyboard'],
    'category': ['Electronics','Books','Electronics','Clothing',None,'Books','Electronics','Clothing','Stationery','Electronics'],
    'price': ['Rs. 899','Rs. 599','Rs. 1299','Rs. 499','Rs. 749',
              'Rs. 899','Rs. 1999','Rs. 2499','Rs. 349','Rs. 3499']
})

print(f"Products: {products.shape[0]} rows x {products.shape[1]} columns")

Products: 10 rows x 4 columns


### Step - 2: Explore the 3 datasets

In [153]:
orders

,order_id,customer_id,product_id,quantity,order_date
0,ORD001,C001,P101,2,2024-01-15
1,ORD002,C002,P102,1,2024-01-15
2,ORD003,C003,P103,3,2024-01-16
3,ORD004,C004,P104,1,2024-01-17
4,ORD005,C005,P105,2,2024-01-17
5,ORD006,C001,P106,1,2024-01-18
6,ORD007,C002,P107,4,2024-01-18
7,ORD008,C003,P108,2,2024-01-19
8,ORD009,NaN,P109,1,2024-01-20
9,ORD010,C005,P110,3,2024-01-20


In [154]:
customers

,customer_id,name,city,segment
0,C001,Priya Sharma,Mumbai,Premium
1,C002,Rahul Verma,Delhi,Regular
2,C003,Anita Desai,Bangalore,Premium
3,C004,VIKRAM PATEL,chennai,Regular
4,C005,sneha iyer,Mumbai,Premium
5,C006,Arjun Nair,KOLKATA,Regular
6,C007,Meera Joshi,pune,Premium
7,C008,Karan Singh,Delhi,Regular
8,C009,Divya Rao,bangalore,Premium
9,C010,Amit Gupta,Mumbai,Regular


In [155]:
products

,product_id,product_name,category,price
0,P101,Wireless Mouse,Electronics,Rs. 899
1,P102,Python Book,Books,Rs. 599
2,P103,USB-C Hub,Electronics,Rs. 1299
3,P104,Cotton T-Shirt,Clothing,Rs. 499
4,P105,Desk Lamp,NaN,Rs. 749
5,P106,Data Science Book,Books,Rs. 899
6,P107,Bluetooth Speaker,Electronics,Rs. 1999
7,P108,Running Shoes,Clothing,Rs. 2499
8,P109,Notebook Set,Stationery,Rs. 349
9,P110,Keyboard,Electronics,Rs. 3499


In [156]:
print("=" * 50)
print("EXPLORING ORDERS DATASET")
print("=" * 50)
print(f"Shape: {orders.shape}")
print("-" * 50)
print("DataTypes: ")
print(orders.dtypes)
print("-" * 50)
print("Missing Values: ")
print(orders.isna().sum())
print("-" * 50)
print(f"Duplicate Records: {orders.duplicated().sum()}")
print("=" * 50)
print()

print("=" * 50)
print("EXPLORING CUSTOMERS DATASET")
print("=" * 50)
print(f"Shape: {customers.shape}")
print("-" * 50)
print("DataTypes: ")
print(customers.dtypes)
print("-" * 50)
print("Missing Values: ")
print(customers.isna().sum())
print("-" * 50)
print(f"Duplicate Records: {customers.duplicated().sum()}")
print("=" * 50)
print()

print("=" * 50)
print("EXPLORING PRODUCTS DATASET")
print("=" * 50)
print(f"Shape: {products.shape}")
print("-" * 50)
print("DataTypes: ")
print(products.dtypes)
print("-" * 50)
print("Missing Values: ")
print(products.isna().sum())
print("-" * 50)
print(f"Duplicate Records: {products.duplicated().sum()}")
print("=" * 50)
print()

EXPLORING ORDERS DATASET
Shape: (21, 5)
--------------------------------------------------
DataTypes: 
order_id         str
customer_id      str
product_id       str
quantity       int64
order_date       str
dtype: object
--------------------------------------------------
Missing Values: 
order_id       0
customer_id    2
product_id     0
quantity       0
order_date     0
dtype: int64
--------------------------------------------------
Duplicate Records: 1

EXPLORING CUSTOMERS DATASET
Shape: (10, 4)
--------------------------------------------------
DataTypes: 
customer_id    str
name           str
city           str
segment        str
dtype: object
--------------------------------------------------
Missing Values: 
customer_id    0
name           0
city           0
segment        0
dtype: int64
--------------------------------------------------
Duplicate Records: 0

EXPLORING PRODUCTS DATASET
Shape: (10, 4)
--------------------------------------------------
DataTypes: 
product_id      

### Step 3: Clean up orders DataFrame - Missing Values, Duplicates, DataTypes

In [157]:
# Handling Missing customer_ids: Dropping rows with missing customer_id
cleaned_orders = orders.copy()
cleaned_orders = cleaned_orders.dropna(subset=['customer_id']).reset_index(drop=True)
print(f"Record Counts: Before Handling Missing Values = {orders.shape[0]}, After Handling Missing Values = {cleaned_orders.shape[0]}")

# Converting order_date to datetime
cleaned_orders['order_date'] = pd.to_datetime(cleaned_orders['order_date'], errors='coerce')

# Handling Duplicate customer_ids: Dropping Rows with Duplicate customer_id with keeping the latest row.
cleaned_orders = cleaned_orders.sort_values(by='order_date', ascending=False).drop_duplicates(subset=['order_id']).reset_index(drop=True)
print(f"Record Counts: Before Dropping Duplicates = {orders.shape[0]}, After Dropping Duplicates = {cleaned_orders.shape[0]}")

orders = cleaned_orders
orders

Record Counts: Before Handling Missing Values = 21, After Handling Missing Values = 19
Record Counts: Before Dropping Duplicates = 21, After Dropping Duplicates = 18


,order_id,customer_id,product_id,quantity,order_date
0,ORD020,C010,P110,2,2024-01-25
1,ORD019,C009,P109,4,2024-01-25
2,ORD018,C008,P108,1,2024-01-24
3,ORD015,C010,P105,3,2024-01-23
4,ORD016,C006,P106,1,2024-01-23
5,ORD013,C008,P103,1,2024-01-22
6,ORD014,C009,P104,2,2024-01-22
7,ORD011,C006,P101,2,2024-01-21
8,ORD012,C007,P102,1,2024-01-21
9,ORD010,C005,P110,3,2024-01-20


### Step 4: Clean up customers DataFrame: Handle Casing and White Spaces

In [158]:
customers_cleaned = customers.copy()
columns_to_clean = ['name', 'city']

for col in columns_to_clean:
    customers_cleaned[col] = customers[col].str.strip().str.title()

## Updated Values:
for col in columns_to_clean:
    print("-" * 50)
    print(f"Updated Values in Column: {col}")
    for initial, updated in zip(customers[col], customers_cleaned[col]):
        if initial != updated:
            print(f"\t '{initial}' -> '{updated}'")
    print("-" * 50)

customers = customers_cleaned
customers

--------------------------------------------------
Updated Values in Column: name
	 ' Anita Desai' -> 'Anita Desai'
	 'VIKRAM PATEL' -> 'Vikram Patel'
	 'sneha iyer' -> 'Sneha Iyer'
--------------------------------------------------
--------------------------------------------------
Updated Values in Column: city
	 ' chennai' -> 'Chennai'
	 'KOLKATA' -> 'Kolkata'
	 'pune' -> 'Pune'
	 'bangalore' -> 'Bangalore'
--------------------------------------------------


,customer_id,name,city,segment
0,C001,Priya Sharma,Mumbai,Premium
1,C002,Rahul Verma,Delhi,Regular
2,C003,Anita Desai,Bangalore,Premium
3,C004,Vikram Patel,Chennai,Regular
4,C005,Sneha Iyer,Mumbai,Premium
5,C006,Arjun Nair,Kolkata,Regular
6,C007,Meera Joshi,Pune,Premium
7,C008,Karan Singh,Delhi,Regular
8,C009,Divya Rao,Bangalore,Premium
9,C010,Amit Gupta,Mumbai,Regular


### Step 5: Clean up products DataFrame: Handling Missing Value and Datatype of price column

In [159]:
products_cleaned = products.copy()

## Filling up missing category with 'Home'
products_cleaned['category'] = products['category'].fillna('Home')
print(f"Missing Categories: Before Handling -> {products['category'].isna().sum()}, After Handling -> {products_cleaned['category'].isna().sum()}")

products_cleaned['price'] = pd.to_numeric(products_cleaned['price'].str.replace('Rs. ', '', regex=False), errors='coerce')
print("Datatypes after converseion:")
print(products_cleaned.dtypes)
print(f"Missing Price Values due to Conversion Error: {products_cleaned['price'].isna().sum()}")
products = products_cleaned

Missing Categories: Before Handling -> 1, After Handling -> 0
Datatypes after converseion:
product_id        str
product_name      str
category          str
price           int64
dtype: object
Missing Price Values due to Conversion Error: 0


### Step 6: Merging Orders, Customers and Products to form Unified Sales Table

In [160]:
orders_with_customer = pd.merge(
    orders,
    customers,
    on='customer_id',
    how='left'
)

unified_sales_report = pd.merge(
    orders_with_customer,
    products,
    on='product_id',
    how='left'
)

unified_sales_report

,order_id,customer_id,product_id,quantity,order_date,name,city,segment,product_name,category,price
0,ORD020,C010,P110,2,2024-01-25,Amit Gupta,Mumbai,Regular,Keyboard,Electronics,3499
1,ORD019,C009,P109,4,2024-01-25,Divya Rao,Bangalore,Premium,Notebook Set,Stationery,349
2,ORD018,C008,P108,1,2024-01-24,Karan Singh,Delhi,Regular,Running Shoes,Clothing,2499
3,ORD015,C010,P105,3,2024-01-23,Amit Gupta,Mumbai,Regular,Desk Lamp,Home,749
4,ORD016,C006,P106,1,2024-01-23,Arjun Nair,Kolkata,Regular,Data Science Book,Books,899
5,ORD013,C008,P103,1,2024-01-22,Karan Singh,Delhi,Regular,USB-C Hub,Electronics,1299
6,ORD014,C009,P104,2,2024-01-22,Divya Rao,Bangalore,Premium,Cotton T-Shirt,Clothing,499
7,ORD011,C006,P101,2,2024-01-21,Arjun Nair,Kolkata,Regular,Wireless Mouse,Electronics,899
8,ORD012,C007,P102,1,2024-01-21,Meera Joshi,Pune,Premium,Python Book,Books,599
9,ORD010,C005,P110,3,2024-01-20,Sneha Iyer,Mumbai,Premium,Keyboard,Electronics,3499


### Step 7: Perform Aggregations

In [161]:
unified_sales_report['total_price'] = unified_sales_report['price'] * unified_sales_report['quantity']

In [162]:
## City wise Summary
city_summary = unified_sales_report.groupby('city').agg(
    total_orders = ('order_id', 'count'),
    total_sales = ('total_price', 'sum'),
    total_quantity = ('quantity', 'sum'),
    average_order_value = ('total_price', 'mean')
)
city_summary

,total_orders,total_sales,total_quantity,average_order_value
city,,,,
Bangalore,4,11289,11,2822.25
Chennai,1,499,1,499.00
Delhi,4,12393,7,3098.25
Kolkata,2,2697,3,1348.50
Mumbai,6,23937,13,3989.50
Pune,1,599,1,599.00


In [163]:
## Category wise Summary
category_summary = unified_sales_report.groupby('category').agg(
    total_orders = ('order_id', 'count'),
    total_sales = ('total_price', 'sum'),
    total_quantity = ('quantity', 'sum'),
    average_order_value = ('total_price', 'mean')
)
category_summary

,total_orders,total_sales,total_quantity,average_order_value
category,,,,
Books,4,2996,4,749.000000
Clothing,4,8994,6,2248.500000
Electronics,7,34283,17,4897.571429
Home,2,3745,5,1872.500000
Stationery,1,1396,4,1396.000000


In [164]:
## Segment wise Summary
segment_summary = unified_sales_report.groupby('segment').agg(
    total_orders = ('order_id', 'count'),
    total_sales = ('total_price', 'sum'),
    total_quantity = ('quantity', 'sum'),
    average_order_value = ('total_price', 'mean')
)
segment_summary

,total_orders,total_sales,total_quantity,average_order_value
segment,,,,
Premium,9,26580,20,2953.333333
Regular,9,24834,16,2759.333333


### Step 8: Pivot Table - City X Category Revenue Summary

In [165]:
city_category_summary = pd.pivot_table(
    unified_sales_report,
    index='city',
    columns='category',
    values='total_price',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total Revenue'
)
city_category_summary

category,Books,Clothing,Electronics,Home,Stationery,Total Revenue
city,,,,,,
Bangalore,0,5996,3897,0,1396,11289
Chennai,0,499,0,0,0,499
Delhi,599,2499,9295,0,0,12393
Kolkata,899,0,1798,0,0,2697
Mumbai,899,0,19293,3745,0,23937
Pune,599,0,0,0,0,599
Total Revenue,2996,8994,34283,3745,1396,51414


<div style="background: #EBF5FB; color: black; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Mini-Project Key Insights:</strong><br><br>
    <strong>Pipeline Summary:</strong><br>
    - Started with 3 messy datasets (21 + 10 + 10 = 41 total rows)<br>
    - Cleaned: 21 → 18 orders (removed 1 duplicate + 2 null customer_ids)<br>
    - Fixed: customer names, city casing, product categories, price format<br>
    - Merged into unified report: 18 rows x 12 columns<br><br>
    <strong>Skills Used:</strong> <code>.drop_duplicates()</code>, <code>.dropna()</code>, <code>.fillna()</code>, <code>.str.strip()</code>, <code>.str.title()</code>, <code>.str.replace()</code>, <code>.astype()</code>, <code>pd.to_datetime()</code>, <code>pd.merge()</code>, <code>.groupby().agg()</code>, <code>pd.pivot_table()</code><br><br>
    <strong>This is the real-world data analysis workflow.</strong> Every professional data analyst does exactly this — clean, merge, analyze — every single day.
</div>